# Load the Existing Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Must use the same model used during ingestion
embedder = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedder,
    collection_name="company-docs"
)

print(f"Chunks in store: {vectorstore._collection.count()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7729.11it/s]


# Basic Similarity Search

In [ ]:
query = "Was war das Hauptziel des Pilotversuchs?"

results = vectorstore.similarity_search(query, k=4)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(f"Text: {doc.page_content[:200]}...")


--- Result 1 ---
Source: chunking.pdf | Page: 2
Text: Die Tabelle zeigt Einzelergebnisse je Massnahme, waehrend die Gesamtquote von 2,2 Prozent aus
dem kombinierten Piloten stammt. Ein Chunk, der nur die Tabelle enthaelt, kann daher scheinbar
widerspruec...

--- Result 2 ---
Source: chunking.pdf | Page: 4
Text: koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle nennt
2,1 Prozent fuer diese Massnahme, die Zielerreichung von 2,2 Prozent bezieht sich jedoch auf
den kombinier...

--- Result 3 ---
Source: chunking.pdf | Page: 1
Text: RAG Chunking Demo
2
1. Ausgangslage und Ziel
Im ersten Quartal lag die Fehlkommissionierungsquote im Lager Nord bei 3,8 Prozent. Besonders
haeufig wurden aehnlich verpackte Artikel verwechselt. Das Pi...

--- Result 4 ---
Source: chunking.pdf | Page: 1
Text: Querverweis: Die konkreten Messwerte stehen in Abschnitt 2. Die visuelle Anordnung des
Lagers ist auf Seite 4 dargestellt....


#  Similarity Search with Scores

In [ ]:
results_with_scores = vectorstore.similarity_search_with_score(query, k=4)

for doc, score in results_with_scores:
    # Collection uses cosine space: score is cosine distance (1 - similarity), lower = more similar
    print(f"Score: {score:.4f} | Source: {doc.metadata.get('source')} | Text: {doc.page_content[:120]}...")

# Metadata Pre-filtering

In [ ]:
# Only search chunks from a specific document category
results_filtered = vectorstore.similarity_search(
    query,
    k=4,
    filter={"document_category": "company-policy"}
)

print(f"Results without filter: 4")
print(f"Results with category filter: {len(results_filtered)}")

for doc in results_filtered:
    print(f"\nSource: {doc.metadata.get('source')} | Category: {doc.metadata.get('document_category')}")
    print(doc.page_content[:200])

Results without filter: 4
Results with category filter: 4

Source: chunking.pdf | Category: company-policy
Die Tabelle zeigt Einzelergebnisse je Massnahme, waehrend die Gesamtquote von 2,2 Prozent aus
dem kombinierten Piloten stammt. Ein Chunk, der nur die Tabelle enthaelt, kann daher scheinbar
widerspruec

Source: chunking.pdf | Category: company-policy
koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle nennt
2,1 Prozent fuer diese Massnahme, die Zielerreichung von 2,2 Prozent bezieht sich jedoch auf
den kombinier

Source: chunking.pdf | Category: company-policy
RAG Chunking Demo
2
1. Ausgangslage und Ziel
Im ersten Quartal lag die Fehlkommissionierungsquote im Lager Nord bei 3,8 Prozent. Besonders
haeufig wurden aehnlich verpackte Artikel verwechselt. Das Pi

Source: chunking.pdf | Category: company-policy
Querverweis: Die konkreten Messwerte stehen in Abschnitt 2. Die visuelle Anordnung des
Lagers ist auf Seite 4 dargestellt.


# 6. Build a Proper RAG System Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_SYSTEM_PROMPT = """You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to answer this."
- Always cite your sources: include the file name and page number for each claim you make.
- Be concise. Do not pad the answer with unnecessary text.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}")
])

print("Prompt template ready.")
print(f"\nSystem message preview:\n{RAG_SYSTEM_PROMPT[:300]}...")

Prompt template ready.

System message preview:
You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to ans...


# Assemble the Full RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Das LLM, das am Ende die Antwort formuliert. temperature=0 -> deterministisch,
# fuer faktenbasierte RAG-Antworten gewollt.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def format_docs(docs):
    """Formatiert die vom Retriever gelieferten Documents zu einem Kontext-String.

    Args:
        docs: Liste von Document-Objekten aus der similarity_search. Jedes hat
            page_content (den Chunk-Text) und metadata (u.a. source und page).

    Returns:
        Einen einzigen String fuer den {context}-Platzhalter im Prompt:
        jeder Chunk beginnt mit seiner Quellenangabe "[Source: <datei>, page <n>]"
        (damit das LLM zitieren kann), die Chunks sind durch "---" getrennt.
    """
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        parts.append(f"[Source: {source}, page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


# Verpackt den Vector Store als Runnable: String rein -> similarity_search
# mit k=4 -> Liste von 4 Documents raus.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# LCEL-Kette (LangChain Expression Language): `|` reicht die Ausgabe eines
# Schritts als Eingabe an den naechsten weiter, wie eine Unix-Pipe.
#
# rag_chain.invoke("Frage") laeuft so durch:
#
# 1. Das Dict ist ein RunnableParallel: BEIDE Zweige bekommen dieselbe Frage.
#      "context":  Frage -> retriever (4 Chunks) -> format_docs (ein String)
#      "question": RunnablePassthrough() reicht die Frage unveraendert durch
#    Ergebnis: {"context": "<4 Chunks als Text>", "question": "<Frage>"}
# 2. | prompt: fuellt die Platzhalter {context} und {question} im Template
#    -> fertige Message-Liste (System + Human)
# 3. | llm: schickt die Messages an gpt-4o-mini -> AIMessage
# 4. | StrOutputParser(): zieht den reinen Text aus der AIMessage
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What is the company refund policy?")
print(answer)

# Ask a Question the Store Can’t Answer

In [20]:
unanswerable = "What is the weather in Berlin today?"
answer_fail = rag_chain.invoke(unanswerable)
print(answer_fail)

I don't have enough information in my knowledge base to answer this.


# Tune k — See the Trade-off

In [22]:
for k in [1, 3, 6]:
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    chain_k = (
        {"context": retriever_k | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    answer_k = chain_k.invoke("Was zeigt die Bildseite?")
    print(f"\n=== k={k} ===")
    print(answer_k[:400])


=== k=1 ===
Die Bildseite zeigt die visuelle Anordnung des Lagers, die auf Seite 4 dargestellt ist (chunking.pdf, Seite 1).

=== k=3 ===
Ich habe nicht genug Informationen in meiner Wissensbasis, um diese Frage zu beantworten.

=== k=6 ===
Ich habe nicht genug Informationen in meinem Wissensstand, um diese Frage zu beantworten.


# Show Retrieved Sources Separately

In [23]:
def answer_with_sources(question, k=4):
    """Beantwortet eine Frage per RAG und listet die verwendeten Quellen auf.

    Args:
        question: Die Nutzerfrage als String.
        k: Anzahl der Chunks, die fuer die Quellen-Anzeige geholt werden.

    Hinweis: Die Chunks werden hier ein zweites Mal geholt, nur um sie anzeigen
    zu koennen — rag_chain macht intern sein eigenes Retrieval mit fixem k=4.
    Bei k != 4 koennen Anzeige und tatsaechlich genutzter Kontext also abweichen.
    """
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)
    answer = rag_chain.invoke(question)

    print("ANSWER:")
    print(answer)
    print("\nSOURCES USED:")
    for doc in docs:
        print(f"  - {doc.metadata.get('source')} (page {doc.metadata.get('page', '?')})")

answer_with_sources("Was zeigt die Bildseite?")

ANSWER:
Die Bildseite zeigt die visuelle Anordnung des Lagers (chunking.pdf, Seite 4).

SOURCES USED:
  - chunking.pdf (page 1)
  - chunking.pdf (page 4)
  - chunking.pdf (page 4)
  - chunking.pdf (page 1)
